# Notebook 8 — Diagnostic de rupture : aspérité unique

**Question :** Les événements observés dans le kymogramme sont-ils de vraies ruptures
(propagation dynamique d'un front de glissement) ou simplement du glissement collectif ?

**Trois diagnostics :**
1. **Profil de déplacement** u_j — sauts discrets avant/pendant/après chaque événement
2. **Stress drop** Δτ_j — chute de contrainte par bloc, distingue blocs rompus (Δτ > 0)
   des blocs voisins chargés (Δτ < 0)
3. **Front de propagation** — carte t_j^onset par événement : ordre temporel des ruptures bloc par bloc

**Cas testés :** ε_asp = −0.3 (barrière VS), ε_asp = 0.5 (référence homogène), ε_asp = 3.0 (aspérité forte)  
**Paramètres :** N=20, ε_bg=0.5, γ_μ=0.5, γ_λ=√0.2, ξ=0.5, f̄=3.2, T=800

In [8]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm, Normalize
import gc

# ── Parameters ──────────────────────────────────────────────────────────────
GM   = 0.5
GL   = np.sqrt(0.2)
XI   = 0.5
EPS  = 0.5
FBAR = 3.2
N    = 20
T    = 800
BURN = 0.3 * T   # burn-in

GM2  = GM**2
GL2  = GL**2

print("Parameters loaded.")

Paramètres chargés.


---
## 0 — Intégrateur et modèle (repris de nb4)

In [9]:
A_tab = np.array([
    [0,0,0,0,0,0,0],[1/5,0,0,0,0,0,0],[3/40,9/40,0,0,0,0,0],
    [44/45,-56/15,32/9,0,0,0,0],[19372/6561,-25360/2187,64448/6561,-212/729,0,0,0],
    [9017/3168,-355/33,46732/5247,49/176,-5103/18656,0,0],
    [35/384,0,500/1113,125/192,-2187/6784,11/84,0]])
B5 = np.array([35/384,0,500/1113,125/192,-2187/6784,11/84,0])
B4 = np.array([5179/57600,0,7571/16695,393/640,-92097/339200,187/2100,1/40])
C  = np.array([0,1/5,3/10,4/5,8/9,1,1])

def rk45_step(f, t, y, h):
    k = np.zeros((7, len(y)))
    k[0] = f(t, y)
    for i in range(1, 7):
        k[i] = f(t + C[i]*h, y + h*np.dot(A_tab[i,:i], k[:i]))
    return y + h*np.dot(B5, k), np.abs(np.dot(B5-B4, k) * h)

def rk45_integrate(f, t_span, y0, rtol=1e-8, atol=1e-10,
                   h_max=0.5, max_steps=50_000_000):
    t0, tf = t_span; y = np.array(y0, dtype=float); t = t0
    h = 1e-3; tl = [t]; yl = [y.copy()]
    for _ in range(max_steps):
        if t >= tf: break
        h = min(h, tf-t)
        y_new, err = rk45_step(f, t, y, h)
        scale = atol + rtol*np.maximum(np.abs(y), np.abs(y_new))
        en = np.sqrt(np.mean((err/scale)**2))
        if en <= 1.0:
            t += h; y = y_new
            tl.append(t); yl.append(y.copy())
            h = min(h * 0.9 * (max(en,1e-30)**-0.2), h_max)
        else:
            h *= 0.9 * en**-0.25
    return np.array(tl), np.array(yl)

def simulate_asp(eps_asp, N=N, T=T):
    eps_arr = np.full(N, EPS); eps_arr[N//2] = eps_asp
    eps1 = 1.0 + eps_arr
    def rhs(t, y):
        u=y[:N]; v=y[N:2*N]; Th=y[2*N:3*N]; dy=np.empty(3*N)
        lap=np.empty(N)
        lap[0]=u[1]-u[0]; lap[1:-1]=u[:-2]-2*u[1:-1]+u[2:]; lap[-1]=u[-2]-u[-1]
        vp1=np.maximum(v+1,1e-30); lv=np.log(vp1)
        dy[:N]=v
        dy[N:2*N]=GM2*lap - GL2*u - (GM2/XI)*(FBAR+Th+lv)
        dy[2*N:3*N]=-vp1*(Th+eps1*lv)   # slip law
        return dy
    u0 = -FBAR*GM2/(XI*GL2)
    xb = (np.arange(1,N+1)-0.5)*(20./N)
    y0 = np.concatenate([u0+np.exp(-(xb-10)**2), np.zeros(N), np.zeros(N)])
    print(f"  Simulating ε_asp={eps_asp}...", end=' ', flush=True)
    t, y = rk45_integrate(rhs, [0,T], y0)
    print(f"done ({len(t)} pts)")
    return t, y[:,:N], y[:,N:2*N], xb

print("Functions defined.")

Fonctions définies.


---
## 1 — Détection d'événements et diagnostics

In [10]:
def compute_stress(u, GM2=GM2, GL2=GL2):
    """Elastic stress τ_j = γ²_μ·Δ_j(u) − γ²_λ·u_j (BC libres)."""
    lap = np.empty_like(u)
    lap[0]=u[1]-u[0]; lap[1:-1]=u[:-2]-2*u[1:-1]+u[2:]; lap[-1]=u[-2]-u[-1]
    return GM2*lap - GL2*u

def detect_events(t, v, t_burn=BURN, v_thresh=1.5, min_gap=5.0):
    """
    Returns list of (i_start, i_peak, i_end) for each event.
    i_start : dernier point à v_max < 0.1 before le pic
    i_peak  : argmax de v_max dans l'event
    i_end   : first point with v_max < 0.1 after the peak
    """
    mask = t >= t_burn
    t_m = t[mask]; v_m = v[mask,:]
    offset = np.where(mask)[0][0]

    vmax = v_m.max(axis=1)
    above = vmax > v_thresh
    events = []
    t_last = -np.inf
    i = 1
    while i < len(above):
        if above[i] and not above[i-1] and (t_m[i]-t_last) >= min_gap:
            # find peak
            i_pk = i
            while i_pk < len(above)-1 and above[i_pk+1]:
                i_pk += 1
            # find start (last v_max < 0.1 before onset)
            i_st = i
            while i_st > 0 and vmax[i_st-1] < 0.1:
                i_st -= 1
            # find end (first v_max < 0.1 after peak)
            i_en = i_pk
            while i_en < len(above)-1 and vmax[i_en] > 0.05:
                i_en += 1
            events.append((offset+i_st, offset+i_pk, offset+i_en))
            t_last = t_m[i_pk]
            i = i_pk + 1
        else:
            i += 1
    print(f"  {len(events)} events detected")
    return events

def propagation_front(t, v, i_start, i_end, v_thresh_front=0.3):
    """
    For each block j, time of first threshold crossing of v_thresh_front
    dans la fenêtre [i_start, i_end]. Retourne array shape (N,) de temps
    (np.nan si le bloc ne dépasse jamais le seuil).
    """
    N = v.shape[1]
    t_onset = np.full(N, np.nan)
    for j in range(N):
        above = np.where(v[i_start:i_end+1, j] > v_thresh_front)[0]
        if len(above) > 0:
            t_onset[j] = t[i_start + above[0]]
    return t_onset

print("Fonctions d'analysis functions defined.")

Fonctions d'analyse définies.


---
## 2 — Simulations

In [11]:
EPS_CASES = {
    '-0.3 (VS barrier)': -0.3,
    '0.5  (reference)':    0.5,
    '3.0  (strong asperity)': 3.0,
}

results = {}
for label, ea in EPS_CASES.items():
    t, u, v, xb = simulate_asp(ea)
    results[label] = {'t': t, 'u': u, 'v': v, 'xb': xb, 'eps_asp': ea}

print("All simulations complete.")

  Simulating ε_asp=-0.3... done (9518 pts)
  Simulating ε_asp=0.5... done (8628 pts)
  Simulating ε_asp=3.0... done (23225 pts)
Toutes les simulations terminées.


---
## 3 — Figures de diagnostic

In [12]:
def rupture_figure(label, data, n_events_show=3):
    """
    4 panels per case :
      (A) Kymogram v̄_j(t)
      (B) Stress drop Δτ_j — tous les events detected (lignes colorées)
          + moyenne en noir épais
      (C) Profil de displacement u_j before / during / after un event type
      (D) Propagation front : heatmap bloc×event du temps normalisé d'onset
    """
    t = data['t']; u = data['u']; v = data['v']; ea = data['eps_asp']
    mask_burn = t >= BURN
    t_m = t[mask_burn]; v_m = v[mask_burn,:]; u_m = u[mask_burn,:]

    events = detect_events(t, v)
    if len(events) == 0:
        print(f"  Aucun event pour {label}")
        return

    blocs = np.arange(1, N+1)

    fig = plt.figure(figsize=(16, 14))
    fig.suptitle(
        rf'Diagnostics de rupture — $
arepsilon_{{asp}}={ea}$  '
        rf'(bloc {N//2+1}/{N}, $
arepsilon_{{bg}}={EPS}$, N={N})',
        fontsize=13, fontweight='bold')
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

    # ── (A) Kymogram ────────────────────────────────────────────────
    ax_A = fig.add_subplot(gs[0, 0])
    idx_plot = np.linspace(0, len(t_m)-1, min(2000,len(t_m)), dtype=int)
    norm_v = TwoSlopeNorm(vmin=-0.3, vcenter=0.5, vmax=2.0)
    im_A = ax_A.imshow(
        v_m[np.ix_(idx_plot, np.arange(N))].T,
        aspect='auto', origin='lower',
        extent=[t_m[idx_plot[0]], t_m[idx_plot[-1]], 0.5, N+0.5],
        cmap='RdBu_r', norm=norm_v)
    ax_A.axhline(N//2+1, color='yellow', lw=1.5, ls='--', label='Asperity')
    ax_A.set_xlabel(r'$\bar{t}$', fontsize=11)
    ax_A.set_ylabel('Block j', fontsize=11)
    ax_A.set_title('(A) Kymogram', fontsize=11, fontweight='bold')
    ax_A.legend(fontsize=8, loc='upper right')
    plt.colorbar(im_A, ax=ax_A, label=r'$\bar{v}_j$', shrink=0.85)

    # ── (B) Stress drop par event ────────────────────────────────
    ax_B = fig.add_subplot(gs[0, 1])
    stress_drops = []
    for i_st, i_pk, i_en in events:
        tau_before = compute_stress(u[i_st, :])
        tau_after  = compute_stress(u[i_en,  :])
        dtau = tau_before - tau_after   # >0 = stress released (rupture)
        stress_drops.append(dtau)
        ax_B.plot(blocs, dtau, alpha=0.25, lw=1, color='steelblue')

    mean_drop = np.mean(stress_drops, axis=0)
    ax_B.plot(blocs, mean_drop, 'k-', lw=2.5, label='Moyenne')
    ax_B.axhline(0, color='gray', lw=0.8, ls='--')
    ax_B.axvline(N//2+1, color='red', lw=1.5, ls='--', alpha=0.7, label='Asperity')
    ax_B.set_xlabel('Block j', fontsize=11)
    ax_B.set_ylabel(r'$\Delta\tau_j = \tau_j^{before} - \tau_j^{after}$', fontsize=10)
    ax_B.set_title(
        f'(B) Stress drop par event ({len(events)} evt.)',
        fontsize=11, fontweight='bold')
    ax_B.legend(fontsize=9)
    ax_B.grid(True, ls=':', alpha=0.4)
    ax_B.set_xticks([1,5,10,15,20])

    # ── (C) Displacement profilement — event représentatif ──────────
    ax_C = fig.add_subplot(gs[1, 0])
    # Choisir l'event médian (ni le premier transient ni le dernier)
    i_ev = len(events) // 2
    i_st, i_pk, i_en = events[i_ev]
    colors_C = ['#1D9E75', '#D85A30', '#378ADD']
    labels_C = [f'Before  (t̄={t[i_st]:.1f})',
                f'Au pic (t̄={t[i_pk]:.1f})',
                f'After  (t̄={t[i_en]:.1f})']
    for arr, c, lbl in zip([u[i_st], u[i_pk], u[i_en]], colors_C, labels_C):
        ax_C.plot(blocs, arr, 'o-', color=c, lw=1.8, ms=5, label=lbl)
    ax_C.axvline(N//2+1, color='red', lw=1.5, ls='--', alpha=0.7, label='Asperity')
    ax_C.set_xlabel('Block j', fontsize=11)
    ax_C.set_ylabel(r'$\bar{u}_j$', fontsize=11)
    ax_C.set_title(f'(C) Profil de displacement — event {i_ev+1}',
                   fontsize=11, fontweight='bold')
    ax_C.legend(fontsize=8)
    ax_C.grid(True, ls=':', alpha=0.4)
    ax_C.set_xticks([1,5,10,15,20])

    # ── (D) Propagation front ─────────────────────────────────────
    ax_D = fig.add_subplot(gs[1, 1])
    # Pour chaque event : temps normalisé d'onset per block
    # t_norm_j = (t_onset_j - t_onset_min) / (t_onset_max - t_onset_min)
    front_matrix = []
    ev_labels = []
    for k_ev, (i_st, i_pk, i_en) in enumerate(events):
        t_onset = propagation_front(t, v, i_st, i_en, v_thresh_front=0.3)
        valid = ~np.isnan(t_onset)
        if valid.sum() < 2:
            continue
        # Normalise entre 0 (premier bloc à rompre) et 1 (dernier)
        t_min = np.nanmin(t_onset); t_max = np.nanmax(t_onset)
        if t_max - t_min < 1e-10:
            t_norm = np.where(valid, 0.5, np.nan)
        else:
            t_norm = (t_onset - t_min) / (t_max - t_min)
        front_matrix.append(t_norm)
        ev_labels.append(f'#{k_ev+1}')

    if len(front_matrix) > 0:
        front_arr = np.array(front_matrix)   # shape (n_ev, N)
        im_D = ax_D.imshow(
            front_arr, aspect='auto', origin='upper',
            extent=[0.5, N+0.5, len(front_matrix)+0.5, 0.5],
            cmap='plasma', vmin=0, vmax=1)
        ax_D.axvline(N//2+1, color='white', lw=1.5, ls='--', alpha=0.8, label='Asperity')
        ax_D.set_xlabel('Block j', fontsize=11)
        ax_D.set_ylabel('Event #', fontsize=11)
        ax_D.set_title('(D) Propagation front (normalised t)',
                       fontsize=11, fontweight='bold')
        ax_D.set_xticks([1,5,10,15,20])
        cb_D = plt.colorbar(im_D, ax=ax_D, shrink=0.85)
        cb_D.set_label('t_onset normalisé (0=premier, 1=dernier)', fontsize=9)
        ax_D.legend(fontsize=9, loc='upper right')
        # Annotation : premier bloc rompu par event
        first_blocs = [np.nanargmin(row)+1 for row in front_matrix]
        frac_left  = np.mean(np.array(first_blocs) <= 2)
        frac_right = np.mean(np.array(first_blocs) >= N-1)
        ax_D.text(0.02, 0.04,
                  f'Nucléation : {frac_left*100:.0f}% bord gauche, '
                  f'{frac_right*100:.0f}% bord droit',
                  transform=ax_D.transAxes, fontsize=8,
                  bbox=dict(fc='white', ec='gray', alpha=0.8))
    else:
        ax_D.text(0.5, 0.5, 'Insufficient events', ha='center',
                  va='center', transform=ax_D.transAxes)

    fname = f'nb8_rupture_eps{str(ea).replace("-","m").replace(".","p")}.png'
    plt.savefig(fname, dpi=140, bbox_inches='tight')
    plt.show()
    print(f"  Figure → {fname}")

print("Function rupture_figure defined.")

Fonction rupture_figure définie.


---
## 4 — Génération des figures

In [13]:
for label, data in results.items():
    print(f"\n{'='*55}")
    print(f"  Cas : {label}")
    print(f"{'='*55}")
    rupture_figure(label, data)


  Cas : -0.3 (barrière VS)
  37 événements détectés


ValueError: 
Diagnostics de rupture — $arepsilon_{asp}=-0.3$  (bloc 11/20, $arepsilon_{bg}=0.5$, N=20)
                         ^
ParseException: Expected end of text, found '$'  (at char 25), (line:1, col:26)

Error in callback <function _draw_all_if_interactive at 0x0000025AE598DB20> (for post_execute), with arguments args (),kwargs {}:


ValueError: 
Diagnostics de rupture — $arepsilon_{asp}=-0.3$  (bloc 11/20, $arepsilon_{bg}=0.5$, N=20)
                         ^
ParseException: Expected end of text, found '$'  (at char 25), (line:1, col:26)

ValueError: 
Diagnostics de rupture — $arepsilon_{asp}=-0.3$  (bloc 11/20, $arepsilon_{bg}=0.5$, N=20)
                         ^
ParseException: Expected end of text, found '$'  (at char 25), (line:1, col:26)

<Figure size 1600x1400 with 6 Axes>

---
## 5 — Grille d'interprétation

| Panneau | Ce qui prouve une rupture | Ce qui indique un simple glissement collectif |
|---|---|---|
| **(B) Stress drop** | Δτ > 0 sur blocs actifs, Δτ < 0 sur voisins non-rompus (transfert) | Δτ uniforme, sans différenciation spatiale |
| **(C) Déplacement** | Saut localisé sur sous-ensemble de blocs ; discontinuité spatiale | Déplacement uniforme sur toute la chaîne |
| **(D) Front** | Gradient clair de t_onset (un bord rompt avant l'autre) | t_onset uniforme (tous les blocs simultanément) |

**Pour ε_asp = −0.3 (barrière VS) :**  
Si c'est une vraie rupture arrêtée par l'aspérité, on attend :
- Δτ > 0 uniquement sur les blocs d'un côté, Δτ ≈ 0 de l'autre côté
- Saut de déplacement limité spatialement à un côté de j* = 10
- Front qui démarre d'un bord et s'arrête à j*

**Pour ε_asp = 3.0 (aspérité forte) :**  
Si c'est un séisme caractéristique, on attend :
- Δτ > 0 fort et localisé sur j* = 10, faible sur les autres
- Saut de déplacement max au bloc central
- t_onset concentré autour de j* (nucléation centrale, propagation vers les bords)